In [2]:
# 6-25-2026

In [3]:
import xarray as xr
import numpy as np
import pandas as pd

In [4]:
ds_path = "../data/seasfire_pyromes_ecoregions.zarr"
domains_path = "../data/ecoregion_domains.zarr"

In [5]:
ds = xr.open_zarr(ds_path, consolidated=True)
domains = xr.open_zarr(domains_path, consolidated=True)

In [7]:
list(ds.data_vars)
# no need for biomes or ecoregion

['area',
 'biomes',
 'cams_co2fire',
 'cams_frpfire',
 'drought_code_max',
 'drought_code_mean',
 'ecoregion',
 'fcci_ba',
 'fcci_ba_valid_mask',
 'fcci_fraction_of_burnable_area',
 'fcci_fraction_of_observed_area',
 'fcci_number_of_patches',
 'fwi_max',
 'fwi_mean',
 'gwis_ba',
 'gwis_ba_valid_mask',
 'lai',
 'lccs_class_1',
 'lccs_class_2',
 'lccs_class_3',
 'lccs_class_4',
 'lccs_class_6',
 'lccs_class_7',
 'lsm',
 'lst_day',
 'ndvi',
 'pop_dens',
 'pyrome',
 'rel_hum',
 'skt',
 'ssr',
 'ssrd',
 'sst',
 'swvl1',
 'swvl2',
 'swvl3',
 'swvl4',
 't2m_max',
 't2m_mean',
 't2m_min',
 'tp',
 'vpd',
 'ws10']

In [8]:
domains

<xarray.Dataset> Size: 2MB
Dimensions:    (latitude: 720, longitude: 1440)
Coordinates:
  * latitude   (latitude) float64 6kB 89.88 89.62 89.38 ... -89.38 -89.62 -89.88
  * longitude  (longitude) float64 12kB -179.9 -179.6 -179.4 ... 179.6 179.9
Data variables:
    domain_id  (latitude, longitude) int16 2MB dask.array<chunksize=(180, 720), meta=np.ndarray>

In [9]:
# drop variables not needed for analysis
ds = ds.drop_vars(["biomes", "ecoregion"])

In [10]:
ds["domain_id"] = domains["domain_id"] # add in domain id to ds, same spatial res

In [11]:
nan_counts_sample = ds.isel(time=0).isnull().sum().compute()
print(nan_counts_sample)

<xarray.Dataset> Size: 344B
Dimensions:                         ()
Coordinates:
    time                            datetime64[ns] 8B 2011-01-01
Data variables: (12/42)
    area                            int64 8B 0
    cams_co2fire                    int64 8B 1031010
    cams_frpfire                    int64 8B 0
    drought_code_max                int64 8B 761095
    drought_code_mean               int64 8B 761095
    fcci_ba                         int64 8B 685679
    ...                              ...
    t2m_mean                        int64 8B 0
    t2m_min                         int64 8B 0
    tp                              int64 8B 0
    vpd                             int64 8B 0
    ws10                            int64 8B 0
    domain_id                       int64 8B 0
Attributes:
    crs:          EPSG:4326
    description:  The SeasFire Cube is a scientific datacube for seasonal fir...
    title:        SeasFire Cube: A Global Dataset for Seasonal Fire Modeling ...


In [ ]:
print(nan_counts_sample.to_array(dim="variable").to_series().sort_values(ascending=False))
# from a sample, the following variables have missing values

variable
cams_co2fire                      1031010
ndvi                               848778
lai                                844949
pop_dens                           770710
fwi_max                            764671
fwi_mean                           764671
drought_code_mean                  761095
drought_code_max                   761095
lst_day                            695812
fcci_number_of_patches             685679
swvl4                              685679
swvl1                              685679
swvl3                              685679
fcci_fraction_of_observed_area     685679
fcci_fraction_of_burnable_area     685679
fcci_ba                            685679
swvl2                              685679
sst                                359785
fcci_ba_valid_mask                      0
cams_frpfire                            0
area                                    0
lccs_class_1                            0
lccs_class_3                            0
lccs_class_2             

In [ ]:
# list of variable stats that will be aggregated for domain descriptor set:
"""
vpd
windspeed
t2m_max
ndvi
tp
fwi_mean
swvl 1 and 4
burned area
frp
# above features will have mean, std, 90th percentile
fire sparsity
fire seasonality
land cover diversity
# above features will have a single value
"""
# although burned area was the target vairable to create the transfer matrix, these variables
#  are meant to characterize regimes, which is independednt of the downstream prediction task
